# Handwritten Meitei Mayek: all experiments of the paper

One notebook for everything the paper reports. If the models already exist
in `WORK` on your Google Drive, they are reused and nothing is retrained.
If they don't, everything is trained from scratch.

| Part | What | A100 time with saved models |
|---|---|---|
| 1 | data split and image cache | 2 min |
| 2 | the ensemble of three fine-tuned CNNs: validation choices, test, error analysis | 5 min |
| 3 | cost (parameters, FLOPs, GPU/CPU latency, memory) and robustness to corrupted scans | 20 min |
| 4 | export the models, check them, publish the Hugging Face demo | 10 min |
| 5 | size-aware variants of the three CNNs | 1–2 h |
| 6 | ablations | about 2 h |
| 7 | parts 3–4 again, if part 5 changed the final system | 0–30 min |
| 8 | figures and `results.json` for the paper | 2 min |

Runtime → Change runtime type → **A100 GPU**, then Runtime → Run all. Every
step saves to Drive, so after a disconnect just run all cells again.
All results end up in `WORK/paper/`.

**Code.** The notebook clones the GitHub repository (falling back to
`MyDrive/Handwritten-Meitei-Mayek-Recognition.zip` if GitHub is unreachable).

**Hugging Face.** Part 4 publishes the trained networks as a model
repository and the demo as a static Space (both free). It needs a Hugging
Face *write* token: add it as a Colab secret called `HF_TOKEN` (key icon on
the left), or paste it when asked.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
FAST = os.environ.get("FAST") == "1"  # tiny CPU smoke test of the code, not a real run

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0.9", "scikit-image", "huggingface_hub",
                    "onnx", "onnxruntime", "onnxscript"], check=True)

REPO_URL = "https://github.com/chingkheinganba231005/Handwritten-Meitei-Mayek-Recognition"
REPO = Path(os.environ.get("REPO", "/content/handwritten-meitei-mayek-recognition"))
CODE_ZIP = Path(os.environ.get("CODE_ZIP", "/content/drive/MyDrive/Handwritten-Meitei-Mayek-Recognition.zip"))
if IN_COLAB:
    import shutil as _sh
    _sh.rmtree(REPO, ignore_errors=True)  # always start from the current code
if not REPO.exists():
    cloned = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO)]).returncode == 0
    if not cloned and CODE_ZIP.exists():  # fallback: the code uploaded to Drive as a zip
        import zipfile as _zf
        _zf.ZipFile(CODE_ZIP).extractall(REPO.parent)
        (REPO.parent / "Handwritten-Meitei-Mayek-Recognition").rename(REPO)
sys.path.insert(0, str(REPO))

DATA = Path(os.environ.get("DATA", "/content/data"))                      # unpacked dataset + split CSVs
CACHE = Path(os.environ.get("CACHE", "/content/cache"))                   # preprocessed image arrays
WORK = Path(os.environ.get("WORK", "/content/drive/MyDrive/tummhcd98"))   # checkpoints and predictions
OUT = WORK / "paper"                                                      # everything the paper uses
HF_ID = os.environ.get("HF_ID", "Chingkheinganba/handwritten-meitei-mayek-recognition")  # model repo and Space
PUBLISH = os.environ.get("PUBLISH", "1") == "1" and not FAST
RUN_SIZE_AWARE = True
RUN_ABLATIONS = True
for folder in (DATA, CACHE, OUT):
    folder.mkdir(parents=True, exist_ok=True)

# Checkpoints and results go to Drive; a full Drive is the usual reason writes fail.
_free = shutil.disk_usage(WORK).free / 1e9 if WORK.exists() else float("inf")
print(f"free space where results are saved ({WORK}): {_free:.1f} GB")
if _free < 5:
    print("WARNING: less than 5 GB free. The remaining runs write about 2 GB; "
          "free up space in Google Drive (and empty its bin) before continuing.")

In [ ]:
import json
import math
import shutil
import time
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from mayek import charset, ensemble as E
from mayek.augment import TTA_VIEWS
from mayek.data import Store, build_cache, load_index
from mayek.model import MEMBERS
from mayek.preprocess import ink_mask, preprocess, stroke_width
from mayek.split import URL, download, make_split
from mayek.train import cached_preds, load_model, predict, train

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

IMG = 32 if FAST else 128
NUM_CLASSES = 55
BASE = {k: dict(v) for k, v in MEMBERS.items()}
if FAST:
    BASE = {k: dict(v, arch="resnet18", epochs=1, pretrained=False, batch=32) for k, v in BASE.items()}
PAIRS = [(44, 25), (46, 9), (11, 47), (33, 34)]

RESULTS = OUT / "results.json"
R = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}


def save():
    RESULTS.write_text(json.dumps(R, indent=2, default=float))


def pct(x):
    return f"{100 * x:.2f}%"

## 1. Data

The split is 15% of every training class for validation, seed 42: 61,504 /
10,826 / 12,794 images. The zip is looked for in `WORK` first, then anywhere
in the top three levels of your Drive, then downloaded from the dataset's
home page. If the university server is down, download
`TUMMHCD-TEST-TRAIN.zip` in a browser from
http://agnigarh.tezu.ernet.in/~sarat/resources.html and put it in `WORK`.

In [ ]:
ZIP = Path(os.environ.get("ZIP", WORK / "TUMMHCD-TEST-TRAIN.zip"))
if not (ZIP.exists() and zipfile.is_zipfile(ZIP)):
    drive_root = Path("/content/drive/MyDrive")
    found = [p for d in ("*", "*/*", "*/*/*") for p in drive_root.glob(f"{d}/TUMMHCD-TEST-TRAIN.zip")]
    found = [p for p in found if zipfile.is_zipfile(p)]
    if found:
        print("using", found[0])
        shutil.copy(found[0], ZIP)
    else:
        print("downloading TUMMHCD ...")
        download(URL, ZIP)

SPLITS, summary = make_split(ZIP, DATA)
print(summary)
INDEX, LABELS, IDX = load_index(SPLITS)
ARRAYS = build_cache(INDEX.file.tolist(), IMG, CACHE)
STORE = Store(ARRAYS, DEVICE, IDX["train"])
va, te = IDX["val"], IDX["test"]
R["data"] = {s: int(len(i)) for s, i in IDX.items()}

# Saved predictions are stored in split order. If the split came out different,
# they would be scrambled, so check before trusting any of them.
REUSED = [n for n in BASE if (WORK / "preds" / f"{n}_dev_val_id.npy").exists()]
for n in REUSED:
    a = E.accuracy(np.load(WORK / "preds" / f"{n}_dev_val_id.npy").astype(np.float32), LABELS[va])
    print(f"saved {n} validation predictions: {pct(a)}")
    if not FAST and a < 0.9:
        raise RuntimeError("Saved predictions don't match this split. Is WORK pointing at the right folder?")
if not FAST and len(REUSED) < len(BASE):
    raise RuntimeError(f"No saved models for {sorted(set(BASE) - set(REUSED))} in {WORK}. "
                       "To train everything from scratch (about 4 extra hours), comment out this check.")

## 2. The three-CNN ensemble

Everything in this part is decided on validation: whether each network uses
test-time augmentation (5 views) and how the three are combined (plain
average or weights fitted to validation log-loss). Only then is the test set
evaluated.

In [ ]:
def member(name, cfg):
    """Validation predictions of the development model, one and five views."""
    def dev_model():
        return train(name, cfg, IDX["train"], "dev", STORE, LABELS, WORK, val_idx=va)
    cache = {}

    def get_model():
        if "m" not in cache:
            cache["m"] = dev_model()
        return cache["m"]
    val = {"id": cached_preds(WORK, f"{name}_dev", "val_id", lambda: predict(get_model(), va, cfg, STORE)),
           "tta": cached_preds(WORK, f"{name}_dev", "val_tta", lambda: predict(get_model(), va, cfg, STORE, TTA_VIEWS))}
    choice = max(("id", "tta"), key=lambda t: E.accuracy(val[t], LABELS[va]))
    return {"name": name, "cfg": cfg, "val": val, "choice": choice,
            "views": TTA_VIEWS if choice == "tta" else ["id"]}


def test_probs(m):
    """Test predictions of the refit model (trained on train + val) with the chosen views."""
    legacy = WORK / "preds" / f"{m['name']}_full_test.npy"  # saved by the original run, same view choice
    if legacy.exists():
        return np.load(legacy).astype(np.float32)

    def make():
        model = train(m["name"], m["cfg"], IDX["full"], "full", STORE, LABELS, WORK)
        return predict(model, te, m["cfg"], STORE, m["views"])
    return cached_preds(WORK, f"{m['name']}_full", f"test_{m['choice']}", make)


def combine_rule(members):
    """Plain average vs log-loss weights, whichever is more accurate on validation (ties: plain)."""
    probs = [m["val"][m["choice"]] for m in members]
    fitted = E.fit_weights(probs, LABELS[va])
    options = {"equal": np.full(len(probs), 1 / len(probs)), "fitted": fitted}
    scores = {k: E.accuracy(E.combine(probs, w), LABELS[va]) for k, w in options.items()}
    rule = "fitted" if scores["fitted"] > scores["equal"] else "equal"
    return rule, options[rule], scores, fitted


base_members = [member(n, c) for n, c in BASE.items()]
rule, weights, scores, fitted = combine_rule(base_members)
BASE_SYS = {"name": "base", "members": base_members, "weights": weights, "rule": rule}
BASE_VAL = E.combine([m["val"][m["choice"]] for m in base_members], weights)

rows = [{"model": m["name"], "views": v, "val acc": E.accuracy(m["val"][v], LABELS[va])}
        for m in base_members for v in ("id", "tta")]
rows += [{"model": "plain average", "views": "chosen", "val acc": scores["equal"]},
         {"model": "log-loss weights " + str(np.round(fitted, 3)), "views": "chosen", "val acc": scores["fitted"]}]
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{100 * v:.2f}"))
print("chosen:", {m["name"]: m["choice"] for m in base_members}, "| combination:", rule)

R["base"] = {
    "val": {m["name"]: {v: E.accuracy(m["val"][v], LABELS[va]) for v in ("id", "tta")} for m in base_members},
    "views": {m["name"]: m["choice"] for m in base_members},
    "val_equal": scores["equal"], "val_fitted": scores["fitted"],
    "fitted_weights": {m["name"]: float(w) for m, w in zip(base_members, fitted)},
    "rule": rule, "weights": {m["name"]: float(w) for m, w in zip(base_members, weights)},
    "val_errors": int((BASE_VAL.argmax(1) != LABELS[va]).sum()),
    "val_confusions": E.confusions(BASE_VAL, LABELS[va], 15),
}
save()

### Does the original size carry information?

A logistic regression on five numbers per image: image height and width,
ink box height and width, ink fraction. Trained on train, scored on
validation, for all classes and for each hard pair against always guessing
the more frequent class of the pair.

In [ ]:
from sklearn.linear_model import LogisticRegression

Z = STORE.meta_z.cpu().numpy()
tr = IDX["train"]
probe = {"all": {"acc": LogisticRegression(max_iter=3000).fit(Z[tr], LABELS[tr]).score(Z[va], LABELS[va]),
                 "chance": 1 / NUM_CLASSES}}
for a, b in PAIRS:
    mt, mv = np.isin(LABELS[tr], [a, b]), np.isin(LABELS[va], [a, b])
    if mv.sum() == 0 or len(set(LABELS[tr][mt])) < 2:
        continue
    clf = LogisticRegression(max_iter=3000).fit(Z[tr][mt], LABELS[tr][mt])
    majority = max(np.mean(LABELS[va][mv] == a), np.mean(LABELS[va][mv] == b))
    probe[f"{a:03d}/{b:03d}"] = {"acc": clf.score(Z[va][mv], LABELS[va][mv]), "majority": majority,
                                 "n_val": int(mv.sum())}
R["size_probe"] = probe
save()
for k, v in probe.items():
    print(k, {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items()})

### Pair specialists

A two-class ConvNeXt-T for each hard pair, consulted only when the
ensemble's top two classes are exactly that pair. `beta` = 0 ignores it,
1 trusts it fully; chosen on validation.

In [ ]:
SPEC_CFG = dict(BASE["convnext_t"], epochs=2 if FAST else 40, lr=1e-4, batch=16 if FAST else 64)


def pair_labels(a, b):
    lab = np.full(len(LABELS), -1)
    lab[LABELS == a], lab[LABELS == b] = 0, 1
    return lab


def specialist(a, b, tag, train_split, eval_idx, eval_split):
    lab = pair_labels(a, b)

    def make():
        tr_idx = IDX[train_split][lab[IDX[train_split]] >= 0]
        v_idx = None if train_split == "full" else va[lab[va] >= 0]
        model = train(f"pair_{a:03d}_{b:03d}", SPEC_CFG, tr_idx, tag, STORE, lab, WORK, val_idx=v_idx, num_classes=2)
        return predict(model, eval_idx, SPEC_CFG, STORE)[:, 0]  # P(class a)
    return cached_preds(WORK, f"pair_{a:03d}_{b:03d}_{tag}", eval_split, make)


SPEC_VAL = {(a, b): specialist(a, b, "dev", "train", va, "val") for a, b in PAIRS}
pair_acc = {}
for (a, b), q in SPEC_VAL.items():
    m = np.isin(LABELS[va], [a, b])
    pair_acc[f"{a:03d}/{b:03d}"] = float(((q[m] > 0.5) == (LABELS[va][m] == a)).mean())
betas = (0, 0.25, 0.5, 0.75, 1.0)
beta_scores = {b: E.accuracy(E.apply_specialists(BASE_VAL, SPEC_VAL, b), LABELS[va]) for b in betas}
BETA = max(beta_scores, key=lambda b: (beta_scores[b], -b))
print("specialist accuracy on its pair:", pair_acc)
print({b: pct(s) for b, s in beta_scores.items()}, "-> beta", BETA)
R["specialists"] = {"pair_acc": pair_acc, "beta_scores": {str(b): s for b, s in beta_scores.items()}, "beta": BETA}
save()

### Test

The single networks and the ensemble as chosen above, on the 12,794 test
images. Nothing after this cell changes the base system.

In [ ]:
def evaluate_test(system, key):
    probs = [test_probs(m) for m in system["members"]]
    final = E.combine(probs, system["weights"])
    if system.get("beta"):
        spec = {(a, b): specialist(a, b, "full", "full", te, "test") for a, b in PAIRS}
        final = E.apply_specialists(final, spec, system["beta"])
    y = LABELS[te]
    correct = int((final.argmax(1) == y).sum())
    out = {"singles": {m["name"]: E.accuracy(p, y) for m, p in zip(system["members"], probs)},
           "acc": correct / len(y), "errors": len(y) - correct, "ci95": E.wilson(correct, len(y)),
           "confusions": E.confusions(final, y, 20),
           "per_class": {f"{c:03d}": v for c, v in E.per_class(final, y).items()}}
    R[key] = out
    save()
    print(pd.Series(out["singles"]).map(pct).to_string())
    print(f"ensemble: {pct(out['acc'])}  ({out['errors']} errors, 95% CI {pct(out['ci95'][0])}-{pct(out['ci95'][1])})")
    print("top confusions:", out["confusions"][:10])
    return final


BASE_SYS["beta"] = BETA
BASE_TEST = evaluate_test(BASE_SYS, "base_test")

## 3. Cost and robustness

**Cost.** Parameters, FLOPs, GPU latency (one image) and throughput (batch
of 256), CPU latency with two threads (the size of a free Hugging Face
Space), preprocessing time, and memory.

**Robustness.** The raw test scans are corrupted before preprocessing:
noise, blur, rotation, lower contrast, uneven lighting, salt-and-pepper
pixels and lower resolution, each at two strengths.

In [ ]:
from functools import partial

from mayek import cost as C
from mayek.robustness import CORRUPTIONS, corrupt_file
from mayek.train import amp_dtype


def measure_cost(system, key):
    if key in R and not os.environ.get("REDO"):  # measured in an earlier session
        print(f"{key}: already measured, skipping (set REDO=1 to repeat)")
        return
    rows, n_img = {}, 64 if FAST else 256
    for m in system["members"]:
        model = load_model(WORK / "runs" / m["name"] / "full" / "final.pt", device=DEVICE)
        path = WORK / "runs" / m["name"] / "full" / "final.pt"
        r = {"params": C.n_params(model), "gflops": C.gflops(model, m["cfg"], IMG), "views": len(m["views"]),
             "file_mb": path.stat().st_size / 1e6}
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()
            r["gpu_ms_1"] = 1000 * C.latency(model, m["cfg"], 1, len(m["views"]), IMG, "cuda")
            t = C.latency(model, m["cfg"], n_img, len(m["views"]), IMG, "cuda", amp=amp_dtype("cuda"))
            r["gpu_img_per_s"] = n_img / t
            r["gpu_peak_mb"] = torch.cuda.max_memory_allocated() / 1e6
        threads = torch.get_num_threads()
        torch.set_num_threads(2)
        r["cpu2_ms_1"] = 1000 * C.latency(model.cpu(), m["cfg"], 1, len(m["views"]), IMG, "cpu",
                                          repeats=3 if FAST else 10, warmup=1 if FAST else 3)
        torch.set_num_threads(threads)
        rows[m["name"]] = r
        del model
    files = INDEX.file.to_numpy()[te[:200]]
    t0 = time.perf_counter()
    for f in files:
        preprocess(np.array(Image.open(f).convert("L")), IMG)
    pre_ms = 1000 * (time.perf_counter() - t0) / len(files)
    total = {"params": sum(r["params"] for r in rows.values()),
             "gflops": sum(r["gflops"] * r["views"] for r in rows.values()),
             "cpu2_ms_1": sum(r["cpu2_ms_1"] for r in rows.values()) + pre_ms, "preprocess_ms": pre_ms}
    if DEVICE == "cuda":
        total["gpu_ms_1"] = sum(r["gpu_ms_1"] for r in rows.values())
        total["gpu_img_per_s"] = 1 / sum(1 / r["gpu_img_per_s"] for r in rows.values())
    R[key] = {"members": rows, "total": total}
    save()
    print(pd.DataFrame(rows).T.round(2).to_string())
    print({k: round(v, 2) for k, v in total.items()})


def measure_robustness(system, key):
    if key in R and not os.environ.get("REDO"):
        print(f"{key}: already measured, skipping (set REDO=1 to repeat)")
        return
    models = [(load_model(WORK / "runs" / m["name"] / "full" / "final.pt", device=DEVICE), m) for m in system["members"]]
    files = INDEX.file.to_numpy()[te].tolist()
    out = {"clean": {"ensemble": R[f"{system['name']}_test"]["acc"],
                     **R[f"{system['name']}_test"]["singles"]}}
    for kind, levels in CORRUPTIONS.items():
        for level in levels:
            arrays = build_cache(files, IMG, CACHE / "corrupt", fn=partial(corrupt_file, kind=kind, level=level),
                                 tag=f"{kind}_{level}_{IMG}")
            store = Store(arrays, DEVICE, stats=STORE.stats, meta_norm=STORE.meta_norm)
            local = np.arange(len(files))
            probs = [predict(model, local, m["cfg"], store, m["views"]) for model, m in models]
            ens = E.combine(probs, system["weights"])
            y = LABELS[te]
            out[f"{kind}@{level}"] = {"ensemble": E.accuracy(ens, y),
                                      **{m["name"]: E.accuracy(p, y) for (_, m), p in zip(models, probs)}}
            print(f"{kind:12s} {level:>5}: ensemble {pct(out[f'{kind}@{level}']['ensemble'])}")
            del store
    R[key] = out
    save()


measure_cost(BASE_SYS, "base_cost")
measure_robustness(BASE_SYS, "base_robustness")

## 4. Release and demo

**Networks.** The refit networks of the final system are exported with
everything inference needs, and checked two ways: on real test scans the
exported recogniser must agree with the evaluation above, and test
characters re-drawn at 384 px with a thin pen check the rescaling the demo
applies to drawings. They are published as a Hugging Face model repository.

**Browser demo.** Hugging Face only hosts *static* Spaces for free, so the
demo runs in the visitor's browser: the EfficientNetV2-S network is
exported to ONNX (weights stored as float16, halving the download) and
checked against PyTorch on test scans. The page reimplements the
preprocessing in JavaScript.

In [ ]:
import onnxruntime as ort

from mayek.augment import view as tta_view
from mayek.recognizer import Recognizer, export
from mayek.web import build_site, export_onnx

MODEL_DIR = OUT / "models"   # -> Hugging Face model repository
DEMO_DIR = OUT / "demo"      # -> static Hugging Face Space


def stroke_ratio(n=2000):
    rng = np.random.default_rng(0)
    ratios = []
    for f in INDEX.file.to_numpy()[rng.choice(IDX["train"], min(n, len(IDX["train"])), replace=False)]:
        gray = np.array(Image.open(f).convert("L"))
        _, ink = ink_mask(gray)
        ratios.append(stroke_width(ink) / max(gray.shape))
    return float(np.median(ratios))


def redraw(gray, pen=10, scale=16):
    """A dataset character as it might look drawn on the demo canvas: thin black strokes on white."""
    from skimage.morphology import skeletonize
    big = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    _, ink = ink_mask(big)
    skel = skeletonize(ink).astype(np.uint8)
    strokes = cv2.dilate(skel, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (pen, pen))) > 0
    return np.where(strokes, 0, 255).astype(np.uint8)


def check_onnx(model, cfg, path, idx):
    """ONNX Runtime vs PyTorch (float32) on the same inputs."""
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    model = model.to(DEVICE).eval()
    agree, diffs = [], []
    for i in range(0, len(idx), 100):
        chunk = idx[i:i + 100]
        x = STORE.normalize(STORE.fetch(chunk, "gray"), "gray")
        meta = STORE.meta(chunk, cfg)
        with torch.no_grad():
            ref = model(x, meta).float().softmax(-1).cpu().numpy()
        feeds = {"image": np.ascontiguousarray(x.float().cpu().numpy())}
        if cfg.get("meta"):
            feeds["meta"] = meta.cpu().numpy()
        out = torch.tensor(sess.run(None, feeds)[0]).softmax(-1).numpy()
        agree.append(out.argmax(1) == ref.argmax(1))
        diffs.append(np.abs(out - ref).max())
    return float(np.concatenate(agree).mean()), float(max(diffs)), sess


def onnx_accuracy(sess, cfg, views, idx):
    probs = []
    for i in range(0, len(idx), 100):
        chunk = idx[i:i + 100]
        raw = STORE.fetch(chunk, "gray")
        p = 0
        for v in views:
            feeds = {"image": np.ascontiguousarray(STORE.normalize(tta_view(raw, v), "gray").float().cpu().numpy())}
            if cfg.get("meta"):
                feeds["meta"] = STORE.meta(chunk, cfg).cpu().numpy()
            p = p + torch.tensor(sess.run(None, feeds)[0]).softmax(-1).numpy()
        probs.append(p / len(views))
    return E.accuracy(np.concatenate(probs), LABELS[idx])


def build_demo(system, key, reference):
    for d in (MODEL_DIR, DEMO_DIR):
        shutil.rmtree(d, ignore_errors=True)
    ratio = stroke_ratio(200 if FAST else 2000)
    test = R[f"{system['name']}_test"]

    # 1. all networks, for Python
    members = [{"name": m["name"], "cfg": m["cfg"], "views": m["views"], "weight": float(w)}
               for m, w in zip(system["members"], system["weights"])]
    export(members, WORK / "runs", MODEL_DIR, STORE.export_stats(), img=IMG, stroke_ratio=ratio)
    cfg_path = MODEL_DIR / "config.json"
    cfg = json.loads(cfg_path.read_text())
    cfg["test_accuracy"] = test["acc"]
    cfg_path.write_text(json.dumps(cfg, indent=2))

    rec = Recognizer(MODEL_DIR, device=DEVICE)
    n = 128 if FAST else 1000
    sub = te[:n]
    grays = [np.array(Image.open(f).convert("L")) for f in INDEX.file.to_numpy()[sub]]
    p = np.concatenate([rec.probs(grays[i:i + 64]) for i in range(0, n, 64)])
    agree = float((p.argmax(1) == reference[:n].argmax(1)).mean())
    drawn = [redraw(g) for g in grays[:300]]
    pd_ = np.concatenate([rec.probs([rec.dataset_like(d, "canvas") for d in drawn[i:i + 64]])
                          for i in range(0, len(drawn), 64)])
    out = {"stroke_ratio": ratio, "agreement_with_eval": agree, "n_checked": n,
           "acc_scans": E.accuracy(p, LABELS[sub]), "acc_redrawn": E.accuracy(pd_, LABELS[sub[:300]]),
           "acc_scans_same_300": E.accuracy(p[:300], LABELS[sub[:300]])}
    print(f"stroke width / image size (median, training scans): {ratio:.3f}")
    print(f"exported networks vs evaluation: {pct(agree)} agreement on {n} test scans")
    print(f"re-drawn characters: {pct(out['acc_redrawn'])} (same 300 scans as they are: {pct(out['acc_scans_same_300'])})")
    if not FAST and agree < 0.99:
        raise RuntimeError("The exported networks disagree with the evaluation; not publishing.")

    # 2. the browser network
    web = next(m for m in system["members"] if m["name"].startswith("effv2_s"))
    model = load_model(WORK / "runs" / web["name"] / "full" / "final.pt", device="cpu")
    (MODEL_DIR / "web").mkdir(parents=True, exist_ok=True)
    onnx_path = MODEL_DIR / "web" / "model.onnx"
    export_onnx(model, web["cfg"], onnx_path, img=IMG)
    onnx_agree, onnx_diff, sess = check_onnx(model, web["cfg"], onnx_path, sub)
    if onnx_agree < (0.9 if FAST else 0.999):  # float16 storage not good enough: keep float32
        export_onnx(model, web["cfg"], onnx_path, img=IMG, half_weights=False)
        onnx_agree, onnx_diff, sess = check_onnx(model, web["cfg"], onnx_path, sub)
    single = test["singles"][web["name"]]
    out.update({"web_network": web["name"], "web_views": web["views"], "web_test_acc_full_set": single,
                "onnx_mb": onnx_path.stat().st_size / 1e6, "onnx_agreement": onnx_agree, "onnx_max_prob_diff": onnx_diff,
                "onnx_acc_on_checked": onnx_accuracy(sess, web["cfg"], web["views"], sub)})
    print(f"browser network {web['name']}: {out['onnx_mb']:.0f} MB, agrees with PyTorch on {pct(onnx_agree)} "
          f"of {n} scans (max prob. difference {onnx_diff:.4f})")
    build_site(DEMO_DIR, web["cfg"], STORE.export_stats(), web["views"], ratio,
               {"model_name": "EfficientNetV2-S", "test_accuracy": single, "ensemble_accuracy": test["acc"],
                "repo_url": REPO_URL, "models_url": f"https://huggingface.co/{HF_ID}"},
               model_url=f"https://huggingface.co/{HF_ID}/resolve/main/web/model.onnx", img=IMG, model_mb=out["onnx_mb"])
    fill = {"{ensemble_accuracy}": f"{100 * test['acc']:.2f}", "{web_accuracy}": f"{100 * single:.2f}",
            "{hf_id}": HF_ID, "{repo_url}": REPO_URL}
    for src, dst in ((REPO / "space" / "README.md", DEMO_DIR / "README.md"),
                     (REPO / "space" / "MODEL_CARD.md", MODEL_DIR / "README.md")):
        text = src.read_text()
        for k, v in fill.items():
            text = text.replace(k, v)
        short = [line for line in text.splitlines() if line.startswith("short_description:")]
        assert all(len(line.split(":", 1)[1].strip()) <= 60 for line in short), "Hugging Face limits this to 60 characters"
        dst.write_text(text)
    R[key] = out
    save()

    fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
    for j in range(8):
        axes[0, j].imshow(drawn[j], cmap="gray")
        axes[1, j].imshow(rec.dataset_like(drawn[j]), cmap="gray")
        for ax in axes[:, j]:
            ax.axis("off")
    axes[0, 0].set_title("re-drawn", loc="left")
    axes[1, 0].set_title("what the networks get", loc="left")
    plt.show()


def publish():
    """Model repository + static Space on Hugging Face. Failures are reported, not fatal."""
    try:
        from huggingface_hub import HfApi, login
        token = None
        if IN_COLAB:
            try:
                from google.colab import userdata
                token = userdata.get("HF_TOKEN")
            except Exception:
                token = None
        if not token:
            from getpass import getpass
            token = getpass("Hugging Face write token (hidden): ")
        login(token=token)
        api = HfApi()
        api.create_repo(HF_ID, repo_type="model", exist_ok=True)
        api.upload_folder(folder_path=MODEL_DIR, repo_id=HF_ID, repo_type="model", commit_message="Trained networks")
        api.create_repo(HF_ID, repo_type="space", space_sdk="static", exist_ok=True)
        api.upload_folder(folder_path=DEMO_DIR, repo_id=HF_ID, repo_type="space", commit_message="Browser demo")
        R["space"] = f"https://huggingface.co/spaces/{HF_ID}"
        R["model_repo"] = f"https://huggingface.co/{HF_ID}"
        save()
        print("demo:", R["space"])
        print("networks:", R["model_repo"])
    except Exception as err:  # keep going: the experiments below don't depend on this
        print(f"Publishing failed ({type(err).__name__}: {err}).\n"
              f"Everything is saved in {MODEL_DIR} and {DEMO_DIR}; re-run this cell later to try again.")


build_demo(BASE_SYS, "base_demo", BASE_TEST)
if PUBLISH:
    publish()

## 5. Size-aware variants

Cropping every character to its ink throws away how large it was written.
The size probe in part 2 shows that for some pairs this is real
information. Here each network also gets the five size numbers through a
small MLP. The final system is whichever candidate is most accurate on
validation: the base ensemble, the size-aware one, or all six networks
together, each with a plain average or fitted weights. Only the winner is
evaluated on test (the base ensemble already was, in part 2).

In [ ]:
FINAL_SYS, FINAL_TEST = BASE_SYS, BASE_TEST
if RUN_SIZE_AWARE:
    size_members = [member(f"{n}_meta", dict(c, meta=True)) for n, c in BASE.items()]
    candidates = {}
    for label, group in (("base", base_members), ("size-aware", size_members), ("all six", base_members + size_members)):
        rule_, w_, sc_, fit_ = combine_rule(group)
        candidates[label] = {"members": group, "weights": w_, "rule": rule_, "scores": sc_, "fitted": fit_}
    table = {k: {"equal": v["scores"]["equal"], "fitted": v["scores"]["fitted"]} for k, v in candidates.items()}
    print(pd.DataFrame(table).T.map(pct).to_string())
    best = max(candidates, key=lambda k: (max(candidates[k]["scores"].values()), k == "base"))
    R["size_aware"] = {
        "val": {m["name"]: {v: E.accuracy(m["val"][v], LABELS[va]) for v in ("id", "tta")} for m in size_members},
        "candidates": table, "chosen": best,
        "fitted_weights": {k: {m["name"]: float(w) for m, w in zip(v["members"], v["fitted"])} for k, v in candidates.items()},
    }
    per_pair = {}
    for label in ("base", "size-aware"):
        probs = E.combine([m["val"][m["choice"]] for m in candidates[label]["members"]], candidates[label]["weights"])
        for a, b in PAIRS:
            mask = np.isin(LABELS[va], [a, b])
            wrong = int(((probs.argmax(1) != LABELS[va]) & mask).sum())
            per_pair.setdefault(f"{a:03d}/{b:03d}", {})[label] = wrong
    R["size_aware"]["val_errors_in_pairs"] = per_pair
    save()
    print("validation errors on images of each pair:", per_pair)
    print("chosen:", best)
    if best != "base":
        c = candidates[best]
        FINAL_SYS = {"name": "final", "members": c["members"], "weights": c["weights"], "rule": c["rule"]}
        final_val = E.combine([m["val"][m["choice"]] for m in c["members"]], c["weights"])
        beta_final = {b: E.accuracy(E.apply_specialists(final_val, SPEC_VAL, b), LABELS[va]) for b in betas}
        FINAL_SYS["beta"] = max(beta_final, key=lambda b: (beta_final[b], -b))
        R["final_system"] = {"candidate": best, "rule": c["rule"], "beta": FINAL_SYS["beta"],
                             "weights": {m["name"]: float(w) for m, w in zip(c["members"], c["weights"])},
                             "views": {m["name"]: m["choice"] for m in c["members"]},
                             "val": max(c["scores"].values()), "val_errors": int((final_val.argmax(1) != LABELS[va]).sum()),
                             "val_confusions": E.confusions(final_val, LABELS[va], 15)}
        save()
        FINAL_TEST = evaluate_test(FINAL_SYS, "final_test")

## 6. Ablations

Each run changes one thing in the ConvNeXt-T recipe (or, for the last
one, the ResNet-50-D input) and is trained on the training split and
scored on validation with a single view, like the reference run. The
second seed of the full recipe also gives the raw weights without
averaging.

In [ ]:
ABLATIONS = {
    "convnext_t_seed1": dict(BASE["convnext_t"], seed=1, keep_raw=True),
    "convnext_t_scratch": dict(BASE["convnext_t"], pretrained=False),
    "convnext_t_64px": dict(BASE["convnext_t"], img=32 if FAST else 64),
    "convnext_t_noaug": dict(BASE["convnext_t"], aug="none"),
    "convnext_t_geomaug": dict(BASE["convnext_t"], aug="geometric"),
    "convnext_t_nosmooth": dict(BASE["convnext_t"], smoothing=0.0),
    "resnet50d_gray": dict(BASE["resnet50d_topo"], channels="gray"),
}
STORES = {IMG: STORE}


def store_for(size):
    if size not in STORES:
        STORES[size] = Store(build_cache(INDEX.file.tolist(), size, CACHE), DEVICE, IDX["train"])
    return STORES[size]


if RUN_ABLATIONS:
    abl = R.get("ablation", {})
    for name, cfg in ABLATIONS.items():
        store = store_for(cfg.get("img", IMG))
        holder = {}

        def model_():
            if "m" not in holder:
                holder["m"] = train(name, cfg, IDX["train"], "dev", store, LABELS, WORK, val_idx=va)
            return holder["m"]
        p = cached_preds(WORK, f"{name}_dev", "val_id", lambda: predict(model_(), va, cfg, store))
        abl[name] = {"val": E.accuracy(p, LABELS[va])}
        if cfg.get("keep_raw"):
            raw_path = WORK / "runs" / name / "dev" / "raw.pt"
            if not raw_path.exists():
                model_()
            q = cached_preds(WORK, f"{name}_dev", "val_raw",
                             lambda: predict(load_model(raw_path, cfg, device=DEVICE), va, cfg, store))
            abl[name]["val_raw"] = E.accuracy(q, LABELS[va])
        hist = torch.load(WORK / "runs" / name / "dev" / "final.pt", map_location="cpu", weights_only=False)["history"]
        abl[name]["history"] = hist
        abl[name]["minutes"] = sum(h["sec"] for h in hist) / 60
        R["ablation"] = abl
        save()
        print(f"{name:22s} {pct(abl[name]['val'])}" + (f"  raw weights {pct(abl[name]['val_raw'])}" if "val_raw" in abl[name] else ""))
    R["ablation_reference"] = {n: R["base"]["val"][n]["id"] for n in ("convnext_t", "resnet50d_topo")}
    save()

## 7. If the final system changed

Cost, robustness and the demo are redone for the size-aware system.

In [ ]:
if FINAL_SYS is not BASE_SYS:
    measure_cost(FINAL_SYS, "final_cost")
    measure_robustness(FINAL_SYS, "final_robustness")
    build_demo(FINAL_SYS, "final_demo", FINAL_TEST)
    if PUBLISH:
        publish()
else:
    print("The base ensemble stays the final system; nothing to redo.")

## 8. Figures and training histories

Saved to `WORK/paper/`. Send `results.json` and the PNG files back to
write the paper from.

In [ ]:
hist = {}
for m in FINAL_SYS["members"] + (base_members if FINAL_SYS is not BASE_SYS else []):
    ck = torch.load(WORK / "runs" / m["name"] / "dev" / "final.pt", map_location="cpu", weights_only=False)
    hist[m["name"]] = ck["history"]
R["histories"] = hist
R["final"] = "final" if FINAL_SYS is not BASE_SYS else "base"
save()

rng = np.random.default_rng(3)
names = {c.index: f"{c.id} {c.display}" for c in charset.CLASSES}

# one sample per class
fig, axes = plt.subplots(5, 11, figsize=(13, 6.4))
for c, ax in zip(range(NUM_CLASSES), axes.flat):
    i = rng.choice(IDX["train"][LABELS[IDX["train"]] == c])
    ax.imshow(np.array(Image.open(INDEX.file[i]).convert("L")), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{c:03d}", fontsize=8)
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "classes.png", dpi=200)
plt.show()

# hard pairs, five samples each
fig, axes = plt.subplots(8, 6, figsize=(6, 8.4))
for r, cls in enumerate([c for p in PAIRS for c in p]):
    pick = rng.choice(IDX["train"][LABELS[IDX["train"]] == cls], 5, replace=False)
    axes[r, 0].text(0.5, 0.5, f"{cls:03d}", ha="center", va="center", fontsize=11)
    for j, i in enumerate(pick):
        axes[r, j + 1].imshow(np.array(Image.open(INDEX.file[i]).convert("L")), cmap="gray", vmin=0, vmax=255)
    for ax in axes[r]:
        ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "pair_samples.png", dpi=200)
plt.show()

# preprocessed inputs
fig, axes = plt.subplots(3, 10, figsize=(15, 4.6))
for col, i in enumerate(rng.choice(IDX["train"], 10, replace=False)):
    for row, arr in enumerate(ARRAYS[:3]):
        axes[row, col].imshow(arr[i], cmap="gray")
        axes[row, col].axis("off")
    axes[0, col].set_title(f"{LABELS[i]:03d}")
fig.tight_layout()
fig.savefig(OUT / "inputs.png", dpi=200)
plt.show()

# validation mistakes of the final system, a random sample
final_val = E.combine([m["val"][m["choice"]] for m in FINAL_SYS["members"]], FINAL_SYS["weights"])
wrong = va[final_val.argmax(1) != LABELS[va]]
show = np.sort(rng.choice(wrong, min(40, len(wrong)), replace=False)) if len(wrong) else []
pos = {g: j for j, g in enumerate(va)}
if len(show):
    fig, axes = plt.subplots(math.ceil(len(show) / 10), 10, figsize=(15, 1.8 * math.ceil(len(show) / 10)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, show):
        ax.imshow(np.array(Image.open(INDEX.file[i]).convert("L")), cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{LABELS[i]:03d}→{final_val[pos[i]].argmax():03d}", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT / "val_mistakes.png", dpi=200)
    plt.show()

print("done. Files in", OUT, ":", sorted(p.name for p in OUT.iterdir()))